# Mind Wandering is Not Monolithic — Results Explorer

This notebook presents the results of our five-phase analysis pipeline without re-running the computation. All outputs are loaded from the `results/` and `figures/` directories generated by `python scripts/run_all.py`.

**Pipeline phases:**
1. **Data Preparation** — Load, harmonize, and filter the EYEMW dataset
2. **Latent Profile Analysis** — Discover MW subtypes via Gaussian Mixture Models
3. **Gaze Discrimination** — Map gaze signatures to profiles with RF and logistic classifiers
4. **Environmental Robustness** — Test stability across lighting, device, and setting
5. **Figures** — Generate publication-ready visualizations

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display, Markdown

RESULTS = Path("results")
FIGURES = Path("figures")

def load_json(name):
    return json.loads((RESULTS / name).read_text())

def show(path, width=700):
    display(Image(filename=str(path), width=width))

---
## Phase 1 — Data Preparation

The EYEMW database contains 22,134 probe-level observations from 20 studies. We filter to 15 eligible studies (those measuring TUT plus at least 2 other MW dimensions) and form two analysis subsets based on shared dimension sets.

In [ ]:
eligibility = load_json("eligibility_info.json")

print(f"Eligible studies: {len(eligibility['eligible_studies'])} of 20")
print(f"  Subset A (video):     Studies {eligibility['subset_a']}  — dims: {eligibility['subset_a_dims']}")
print(f"  Subset B (listening):  Studies {eligibility['subset_b']} — dims: {eligibility['subset_b_dims']}")

### Eligibility matrix

Which MW dimensions are available in each study? Intentionality is entirely missing across all 20 studies.

In [ ]:
eligibility_matrix = pd.read_csv(RESULTS / "eligibility_matrix.csv", index_col=0)
eligibility_matrix.style.map(lambda v: "background-color: #c6efce" if v else "background-color: #ffc7ce")

In [ ]:
show(FIGURES / "fig1_eligibility_matrix.png")

### Harmonized dataset overview

In [ ]:
provenance = load_json("provenance_log.json")
harmonized = pd.read_parquet(RESULTS / "harmonized_data.parquet")

print(f"Raw rows: {provenance['total_rows_original']:,}")
print(f"After eligibility filtering: {len(harmonized):,} rows × {harmonized.shape[1]} columns")
print(f"\nStudies retained: {harmonized['StudyNum'].nunique()}")
print(f"Unique participants: {harmonized['ParticipantNum'].nunique()}")

---
## Phase 2 — Latent Profile Analysis

We fit Gaussian Mixture Models with k = 2–6 components on each subset's MW dimensions (harmonized to 0–1 scale). BIC selects the optimal number of profiles, and bootstrap resampling (n = 1,000) tests stability.

### Model selection

In [ ]:
for subset in ["SubsetA", "SubsetB"]:
    display(Markdown(f"**{subset}**"))
    ms = pd.read_csv(RESULTS / f"lpa_model_selection_{subset}.csv")
    display(ms.style.format({"BIC": "{:,.0f}", "AIC": "{:,.0f}", "entropy": "{:.4f}",
                             "avg_posterior": "{:.4f}", "log_likelihood": "{:,.0f}"}))

In [ ]:
show(FIGURES / "model_selection_SubsetA.png")
show(FIGURES / "model_selection_SubsetB.png")

### Bootstrap stability

In [ ]:
for subset in ["SubsetA", "SubsetB"]:
    b = load_json(f"bootstrap_results_{subset}.json")
    display(Markdown(f"**{subset}** (k = {b['best_k']})"))
    print(f"  ARI:       {b['mean_ari']:.2f}  (95% CI: {b['ari_ci_lower']:.2f}–{b['ari_ci_upper']:.2f})")
    print(f"  Posterior: {b['mean_posterior']:.3f}  (95% CI: {b['posterior_ci_lower']:.3f}–{b['posterior_ci_upper']:.3f})")
    print(f"  Meets ≥0.70 posterior target: {b['meets_posterior_target']}")
    print()

### Discovered profiles

Five profiles emerge in each subset, characterized by distinct combinations of MW dimension levels.

In [ ]:
profile_labels_a = {
    0: "MW",
    1: "On-Task + Positive",
    2: "On-Task",
    3: "On-Task + Disengaged",
    4: "MW + Disengaged",
}
profile_labels_b = {
    0: "On-Task + Aware",
    1: "MW",
    2: "MW + Disengaged + Aware",
    3: "On-Task + Aware + FMT",
    4: "On-Task",
}

for subset, labels in [("SubsetA", profile_labels_a), ("SubsetB", profile_labels_b)]:
    display(Markdown(f"**{subset}**"))
    ps = pd.read_csv(RESULTS / f"profile_stats_{subset}.csv")
    ps.insert(1, "label", ps["profile"].map(labels))
    # Format means to 2 decimal places, keep pct as integer
    fmt = {c: "{:.2f}" for c in ps.columns if c.endswith(("_mean", "_sd"))}
    fmt["avg_posterior"] = "{:.4f}"
    fmt["pct"] = "{:.1f}%"
    display(ps.style.format(fmt))
    print()

In [ ]:
show(FIGURES / "fig3_radar_profiles_SubsetA.png")
show(FIGURES / "fig3_radar_profiles_SubsetB.png")

### MW dimension correlations

Low-to-moderate correlations between MW dimensions confirm they are not redundant — justifying the multidimensional approach.

In [ ]:
show(FIGURES / "fig2_correlation_subset_A.png")
show(FIGURES / "fig2_correlation_subset_B.png")

---
## Phase 3 — Gaze-Based Profile Discrimination

Can gaze behavior alone distinguish these MW profiles? We train Random Forest and logistic regression classifiers (5-fold CV, multiclass one-vs-rest AUROC) using 8 gaze features.

### Classifier performance

In [ ]:
summary = load_json("phase3_summary.json")

perf = pd.DataFrame({
    "Metric": ["Logistic AUROC", "RF AUROC", "Binary AUROC (on-task vs. MW)"],
    "Subset A": [
        f"{summary['SubsetA']['logistic_auroc']:.3f}",
        f"{summary['SubsetA']['rf_auroc']:.3f}",
        f"{summary['SubsetA']['binary_comp']['binary_auroc']:.3f}",
    ],
    "Subset B": [
        f"{summary['SubsetB']['logistic_auroc']:.3f}",
        f"{summary['SubsetB']['rf_auroc']:.3f}",
        f"{summary['SubsetB']['binary_comp']['binary_auroc']:.3f}",
    ],
})
display(perf.style.hide(axis="index"))

Multi-profile AUROC (~0.59) is modest but above chance (0.50). The binary baseline (on-task vs. any MW) outperforms, reflecting the difficulty of distinguishing *between* MW subtypes from gaze alone.

### Feature importance

In [ ]:
for subset in ["SubsetA", "SubsetB"]:
    display(Markdown(f"**{subset}** — RF Gini importance"))
    fi = pd.read_csv(RESULTS / f"feature_importance_{subset}.csv")
    fi = fi.sort_values("rf_importance", ascending=False)
    display(fi.style.format({"rf_importance": "{:.3f}"}).hide(axis="index"))
    print()

In [ ]:
show(FIGURES / "fig4_shap_SubsetA.png")
show(FIGURES / "fig4_shap_SubsetB.png")

### Confusion matrices

In [ ]:
show(FIGURES / "confusion_matrix_SubsetA.png")
show(FIGURES / "confusion_matrix_SubsetB.png")

---
## Phase 4 — Environmental Robustness (Slicing Analysis)

Do profiles and gaze discrimination hold up across different recording environments? We slice by **lighting** (well-lit, dim, none), **device** (computer, laptop), and **setting** (home, public) and examine within-slice AUROC and feature-importance rank stability.

### Within-slice AUROC

In [ ]:
for subset in ["SubsetA", "SubsetB"]:
    display(Markdown(f"**{subset}**"))
    rt = pd.read_csv(RESULTS / f"robustness_table_{subset}.csv")
    display(rt.style.format({"auroc": "{:.3f}", "accuracy": "{:.3f}"}).hide(axis="index"))

In [ ]:
show(FIGURES / "fig5_robustness_heatmap_SubsetA.png")
show(FIGURES / "fig5_robustness_heatmap_SubsetB.png")

### Feature importance stability across slices

Spearman rank correlations of feature importance between slice pairs. Values above 0.70 indicate robust feature rankings.

In [ ]:
for subset in ["SubsetA", "SubsetB"]:
    display(Markdown(f"**{subset}**"))
    ma = pd.read_csv(RESULTS / f"moderation_analysis_{subset}.csv")
    display(ma.style.format({"spearman_rho": "{:.2f}", "p_value": "{:.4f}"}).hide(axis="index"))

Feature importance rankings are robust across **lighting** (rho = 0.71–0.86) and **device type** (rho = 0.88), but sensitive to **experimental setting** (rho = 0.14). This suggests gaze-based MW detection generalizes across hardware and illumination conditions but may not transfer well between home and public environments.

---
## Summary

Key findings at a glance:

In [ ]:
ps = load_json("paper_summary.json")

summary_table = pd.DataFrame({
    "Metric": [
        "Observations",
        "Profiles discovered",
        "Bootstrap ARI",
        "RF AUROC (gaze only)",
        "Binary AUROC (on-task vs. MW)",
        "Slice AUROC range",
        "Feature importance stability (lighting)",
        "Feature importance stability (device)",
    ],
    "Subset A (video, 9 studies)": [
        f"{int(ps['SubsetA_total_n']):,}",
        str(ps["SubsetA_n_profiles"]),
        f"{ps['SubsetA_bootstrap_ari']:.2f}",
        f"{ps['SubsetA_rf_auroc']:.3f}",
        f"{ps['SubsetA_binary_comp']['binary_auroc']:.3f}",
        f"{ps['SubsetA_min_slice_auroc']:.2f}–{ps['SubsetA_max_slice_auroc']:.2f}",
        "rho = 0.71–0.86",
        "rho = 0.88",
    ],
    "Subset B (listening, 2 studies)": [
        f"{int(ps['SubsetB_total_n']):,}",
        str(ps["SubsetB_n_profiles"]),
        f"{ps['SubsetB_bootstrap_ari']:.2f}",
        f"{ps['SubsetB_rf_auroc']:.3f}",
        f"{ps['SubsetB_binary_comp']['binary_auroc']:.3f}",
        f"{ps['SubsetB_min_slice_auroc']:.2f}–{ps['SubsetB_max_slice_auroc']:.2f}",
        "rho = 0.71",
        "—",
    ],
})

display(summary_table.style.hide(axis="index"))

### Takeaways

1. **Mind wandering is not monolithic.** LPA consistently identifies 5 distinct profiles in both subsets, with high posterior probabilities and strong bootstrap stability (ARI = 0.82 in Subset A).

2. **Gaze signals partially discriminate profiles.** Multi-profile AUROC of 0.59 is modest but consistent. The binary on-task-vs-MW framing reaches 0.70, suggesting gaze captures the *presence* of MW better than its *flavor*.

3. **Feature importance generalizes across hardware and lighting.** Spearman correlations of 0.71–0.88 across lighting and device slices indicate that the same gaze features matter regardless of recording conditions — an encouraging sign for real-world deployment.

4. **Home vs. public settings remain a challenge.** Feature importance drops to rho = 0.14 across settings, suggesting environmental context modulates gaze-MW relationships in ways that current features don't capture.